# SadaLLM (صدى): بناء نموذج لغوي مصغر للأغراض التعليمية من الصفر (10.6M Parameters)
### *شرح معمارية المحولات (Transformers) والأوراق البحثية التأسيسية — مع تطبيق عملي على نصوص من نظام العمل السعودي*

**صدى (SadaLLM)** هو **نموذج لغوي مصغر ومفتوح للأغراض التعليمية (Educational Micro-LLM)** تم بناؤه من الصفر لتعليم الطلاب الأسس الرياضية والهندسية للنماذج اللغوية التوليدية بدون أي مكتبات مغلقة.

نستخدم في هذا الدرس نصوصاً مختارة من **نظام العمل السعودي** كمجرد **مثال تطبيقي (Case Study)** لملاحظة كيف يتعلم النموذج التراكيب اللغوية العربية، وكيف يربط السؤال بالجواب، وكيف يولّد الكلمات تلقائياً خطوة بخطوة على المعالج (**CPU**).

---

### خارطة الأوراق البحثية التأسيسية للمشروع

| المكون الهندسي | الورقة البحثية التأسيسية | الفكرة الجوهرية |
| :--- | :--- | :--- |
| **1. التضمين المتجهي (Embeddings)** | [Bengio et al. (2003)](https://www.jmlr.org/papers/volume3/bengio03a/bengio03a.pdf) & [Mikolov (2013)](https://arxiv.org/abs/1310.4546) | تحويل الرموز والمحارف المنفصلة إلى فضاء متجهي مستمر $\mathbb{R}^d$ |
| **2. الترميز الموضعي (Positional)** | [Vaswani et al. (2017)](https://arxiv.org/abs/1706.03762) & [Gehring (2017)](https://arxiv.org/abs/1705.03122) | حقن ترتيب المواقع لكسر خاصية عدم التأثر بالترتيب |
| **3. آلية الانتباه السببي** | [Bahdanau (2014)](https://arxiv.org/abs/1409.0473) $\to$ [Luong (2015)](https://arxiv.org/abs/1508.04025) $\to$ [Vaswani (2017)](https://arxiv.org/abs/1706.03762) | حساب الانتباه المقيّس مع القناع السببي Causal Mask |
| **4. استقرار التدريب (Pre-LN & GELU)** | [Ba (2016)](https://arxiv.org/abs/1607.06450) & [Hendrycks (2016)](https://arxiv.org/abs/1606.08415) | تدفق التدرجات عبر الـ Residual Stream وتنشيط GELU |
| **5. مشاركة الأوزان (Weight Tying)** | [Press & Wolf (2017)](https://arxiv.org/abs/1608.05859) | توحيد مصفوفة التضمين مع رأس التوقع لتقليل استهلاك الذاكرة |
| **6. التوليد والاعتيان (Sampling)** | [Radford (2018) - GPT](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) & [Fan (2018)](https://arxiv.org/abs/1805.04833) | التوليد التلقائي مع التحكم في العشوائية عبر الحرارة و Top-K |

In [ ]:
import math
import time
import random
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# ضبط البذور لضمان تطابق النتائج
torch.manual_seed(42)
random.seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"إصدار PyTorch: {torch.__version__}")
print(f"جهاز التشغيل لنموذج صدى التعليمي (SadaLLM): {device.upper()}")

---
## القسم 1: بيانات المثال التطبيقي (نظام العمل السعودي) ومُرمز المحارف

نعتمد **الترميز المباشر للحروف العربية (Arabic Character Tokenizer)** بحيث يمثل كل حرف عربي توكن مستقل كامل ($V \approx 75$)، مما يوضح للطلاب كيف يتعلم النموذج بناء الكلمات والجمل حرفاً بحرف دون تلعثم:

In [ ]:
SAUDI_LABOR_LAW_TEXT = """
الباب الأول: التعريفات والأحكام العامة
المادة الأولى: يقصد بالألفاظ والعبارات الآتية المعاني المبينة أمامها:
الوزارة: وزارة الموارد البشرية والتنمية الاجتماعية.
صاحب العمل: كل شخص ذي صفة طبيعية أو اعتبارية يشغل عاملاً أو أكثر لقاء أجر.
العامل: كل شخص ذي صفة طبيعية يعمل لمصلحة صاحب العمل وتحت إدارته أو إشرافه مقابل أجر.
الأجر الأساسي: كل ما يعطى للعامل مقابل عمله بموجب عقد عمل مكتوب أو غير مكتوب دون احتساب البدلات أو المزايا الأخرى.
الأجر الفعلي: الأجر الأساسي مضافاً إليه سائر الزيادات المستحقة والبدلات المقررة للعامل لقاء جهد أو مخاطر.

الباب الثاني: عقد العمل وفترة التجربة
المادة الخمسون: عقد العمل هو عقد مبرم بين صاحب عمل وعامل، يتعهد الأخير بموجبه أن يعمل تحت إدارة صاحب العمل أو إشرافه مقابل أجر.
المادة الثالثة والخمسون: إذا كان العامل خاضعاً لفترة تجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديدها بوضوح بحيث لا تزيد على تسعين يوماً.
يجوز باتفاق مكتوب بين العامل وصاحب العمل تمديد فترة التجربة على ألا تزيد في مجموعها على مائة وثمانين يوماً.
لا تدخل في حساب فترة التجربة إجازة عيدي الفطر والأضحى والإجازة المرضية.

الباب الثالث: ساعات العمل والراحة الأسبوعية
المادة الثامنة والتسعون: لا يجوز تشغيل العامل تشغيلاً فعلياً أكثر من ثماني ساعات في اليوم الواحد إذا اعتمد صاحب العمل المعيار اليومي، أو أكثر من ثمانٍ وأربعين ساعة في الأسبوع إذا اعتمد المعيار الأسبوعي.
تخفض ساعات العمل الفعلية خلال شهر رمضان للمسلمين بحيث لا تزيد على ست ساعات في اليوم، أو ست وثلاثين ساعة في الأسبوع.
المادة المائة وواحد: الجمعة هو يوم الراحة الأسبوعية لجميع العمال بأجر كامل.

الباب الرابع: الإجازات الرسمية والسنوية
المادة التاسعة بعد المائة: يستحق العامل عن كل عام إجازة سنوية لا تقل مدتها عن واحد وعشرين يوماً، وتزاد إلى مدة لا تقل عن ثلاثين يوماً إذا أمضى العامل في خدمة صاحب العمل خمس سنوات متصلة، وتكون الإجازة بأجر يدفع مقدماً.
المادة الثالثة عشرة بعد المائة: يستحق العامل إجازة بأجر كامل لمدة خمسة أيام في حال وفاة زوجه أو أحد أصوله أو فروعه، أو عند زواجه، وثلاثة أيام في حال ولادة مولود له.

الباب الخامس: انتهاء العقد ومكافأة نهاية الخدمة
المادة الرابعة والثمانون: إذا انتهت علاقة العمل وجب على صاحب العمل أن يدفع إلى العامل مكافأة عن مدة خدمته تحسب على أساس أجر نصف شهر عن كل سنة من السنوات الخمس الأولى، وأجر شهر عن كل سنة من السنوات التالية، ويتخذ الأجر الأخير أساساً لحساب المكافأة.

صيغ الأسئلة والأجوبة القانونية لنظام العمل:
سؤال: كم الحد الأقصى لساعات العمل اليومية في نظام العمل السعودي؟
الجواب: لا يجوز تشغيل العامل تشغيلاً فعلياً أكثر من ثماني ساعات في اليوم الواحد، وتخفض في رمضان إلى ست ساعات.

سؤال: كم تبلغ فترة التجربة في نظام العمل؟
الجواب: الأصل ألا تزيد فترة التجربة على تسعين يوماً، ويجوز باتفاق الطرفين تمديدها بشرط ألا تتجاوز مائة وثمانين يوماً.

سؤال: كيف تحسب مكافأة نهاية الخدمة للعامل؟
الجواب: تحسب المكافأة على أساس أجر نصف شهر عن كل سنة من السنوات الخمس الأولى، وأجر شهر عن كل سنة من السنوات التالية بناءً على الأجر الأخير.

سؤال: كم يستحق العامل إجازة سنوية في نظام العمل السعودي؟
الجواب: يستحق العامل إجازة سنوية لا تقل عن واحد وعشرين يوماً، وتزاد إلى ثلاثين يوماً إذا أمضى خمس سنوات متصلة في الخدمة.
"""

# بناء قاموس الحروف العربية
ARABIC_CHARS = sorted(list(set(
    "ابتثجحخدذرزسشصضطظعغفقكلمنهويىةءئؤإأآ"
    "0123456789"
    " \n\t،؛.:!؟-\"'/()[]"
)))
VOCAB = ARABIC_CHARS + ['<pad>', '<eos>']
VOCAB_SIZE = len(VOCAB)
CHAR2IDX = {ch: i for i, ch in enumerate(VOCAB)}
IDX2CHAR = {i: ch for i, ch in enumerate(VOCAB)}

def encode_text(text: str) -> list[int]:
    return [CHAR2IDX.get(ch, CHAR2IDX['<pad>']) for ch in text]

def decode_ids(ids: list[int]) -> str:
    return ''.join([IDX2CHAR[i] for i in ids if i < len(VOCAB) and IDX2CHAR[i] not in ['<pad>', '<eos>']])

corpus = (SAUDI_LABOR_LAW_TEXT.strip() + "\n\n") * 20
data_tokens = torch.tensor(encode_text(corpus), dtype=torch.long)
print(f"حجم قاموس الحروف العربية: {VOCAB_SIZE} حرف وعلامة")
print(f"إجمالي توكنات التدريب للمثال التطبيقي: {len(data_tokens):,} حرف")

---
## القسم 2: معمارية نموذج صدى التعليمي - SadaLLM (10.6M Parameters)

تتكون المعمارية من 6 طبقات Transformer مكدسة، ببعد تمثيل $d = 384$ و 6 رؤوس انتباه، مع تنشيط GELU ومشاركة الأوزان (Weight Tying):

In [ ]:
@dataclass
class ModelConfig:
    vocab_size: int = VOCAB_SIZE
    max_context_len: int = 256  # 256 حرفاً عربياً متصلاً
    d_model: int = 384
    n_layers: int = 6
    n_heads: int = 6
    d_ff: int = 1536
    dropout: float = 0.0
    tie_weights: bool = True

cfg = ModelConfig()

class LayerNorm(nn.Module):
    def __init__(self, d_model: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(d_model))
        self.beta = nn.Parameter(torch.zeros(d_model))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        return self.gamma * (x - mean) / torch.sqrt(var + self.eps) + self.beta


class CausalSelfAttention(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.d_model = cfg.d_model
        self.n_heads = cfg.n_heads
        self.d_k = cfg.d_model // cfg.n_heads

        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.k_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.v_proj = nn.Linear(cfg.d_model, cfg.d_model)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model)

        mask = torch.triu(torch.full((cfg.max_context_len, cfg.max_context_len), float('-inf')), diagonal=1)
        self.register_buffer('mask', mask)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_k).transpose(1, 2)

        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.d_k)
        scores = scores + self.mask[:T, :T]
        weights = F.softmax(scores, dim=-1)

        out = weights @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.ln1 = LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg)
        self.ln2 = LayerNorm(cfg.d_model)
        self.mlp = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_ff),
            nn.GELU(),
            nn.Linear(cfg.d_ff, cfg.d_model)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x


class SadaLLM(nn.Module):
    def __init__(self, cfg: ModelConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Embedding(cfg.max_context_len, cfg.d_model)
        self.blocks = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = LayerNorm(cfg.d_model)
        self.lm_head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        if cfg.tie_weights:
            self.lm_head.weight = self.tok_emb.weight

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.lm_head(x)

model = SadaLLM(cfg).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"إجمالي المعاملات القابلة للتدريب في نموذج صدى التعليمي (SadaLLM): {total_params:,} معامل (~10.6M)")

---
## القسم 3: حلقة التدريب لنموذج صدى (SadaLLM Training)

ندرب النموذج على نصوص المثال التطبيقي لنراقب كيف تنخفض دالة الخسارة (Cross-Entropy Loss):

In [ ]:
def get_batch(tokens: torch.Tensor, batch_size: int = 16, context_len: int = 256):
    ix = torch.randint(0, len(tokens) - context_len - 1, (batch_size,))
    x = torch.stack([tokens[i:i + context_len] for i in ix]).to(device)
    y = torch.stack([tokens[i + 1:i + 1 + context_len] for i in ix]).to(device)
    return x, y

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
loss_history = []
start_time = time.time()
num_steps = 450

print("بدء تدريب نموذج صدى التعليمي (SadaLLM)...")
print("-" * 55)
model.train()
for step in range(1, num_steps + 1):
    xb, yb = get_batch(data_tokens, batch_size=16, context_len=cfg.max_context_len)
    logits = model(xb)
    loss = F.cross_entropy(logits.reshape(-1, cfg.vocab_size), yb.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    loss_history.append(loss.item())
    if step % 50 == 0 or step == 1:
        print(f"الخطوة {step:>3} | الخسارة (Loss): {loss.item():.4f} | الوقت: {time.time() - start_time:.1f}s")

# رسم بياني لهبوط دالة الخسارة
plt.figure(figsize=(7, 3.5))
plt.plot(loss_history, color='forestgreen', lw=2)
plt.title("SadaLLM (صدى) - منحنى تقارب الخسارة أثناء التعلم")
plt.xlabel("خطوات التحسين (Step)")
plt.ylabel("دالة الخسارة (Cross-Entropy Loss)")
plt.grid(True, alpha=0.3)
plt.show()

---
## القسم 4: اختبار التوليد التلقائي (Autoregressive Inference)

نختبر الآن كيف تعلم النموذج استرجاع وتوليد الإجابات حرفاً تلو الآخر:

In [ ]:
def generate_text(model: SadaLLM, prompt: str, max_new_tokens: int = 140, temperature: float = 0.2, top_k: int = 5) -> str:
    model.eval()
    tokens = encode_text(prompt)
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = tokens[-cfg.max_context_len:]
            x = torch.tensor([idx_cond], dtype=torch.long, device=device)
            logits = model(x)[0, -1, :] / max(temperature, 1e-5)

            if top_k is not None and top_k > 0:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[-1]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()
            tokens.append(next_token)

    return decode_ids(tokens)

sample_prompts = [
    "سؤال: كم الحد الأقصى لساعات العمل اليومية في نظام العمل السعودي؟\nالجواب:",
    "المادة الثالثة والخمسون: إذا كان العامل خاضعاً لفترة تجربة",
    "سؤال: كيف تحسب مكافأة نهاية الخدمة للعامل؟\nالجواب:"
]

for q in sample_prompts:
    print("=" * 65)
    print(f"[المدخل]:\n{q}")
    ans = generate_text(model, prompt=q)
    print(f"\n[توليد نموذج صدى / SadaLLM]:\n{ans}")
    print("=" * 65)

---
## القسم 5: مساحة التجربة للطلاب (SadaLLM Playground)

يمكن للطلاب تجربة تغيير المدخلات أو تغيير قيمة الحرارة (`temperature`) لملاحظة تأثيرها على التوليد:

In [ ]:
student_prompt = "سؤال: كم يستحق العامل إجازة سنوية في نظام العمل السعودي؟\nالجواب:"
response = generate_text(model, prompt=student_prompt, max_new_tokens=120, temperature=0.2, top_k=5)

print(response)